In [1]:
import sys
import pandas as pd
import numpy as np

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Jupyter notebook is working!")

Python: 3.13.15
Pandas: 3.1.0
NumPy: 2.5.4
Jupyter notebook is working!


In [2]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Jupyter is working successfully! 🚀")

Pandas: 3.1.0
NumPy: 2.5.4
Jupyter is working successfully! 🚀


# E-commerce Dataset Exploration

## Initial Dataset Inspection

**Dataset:** eCommerce Behavior Data from Multi Category Store

**Source:** Kaggle / Open CDP

**Selected period:** October 2019

**Objective:** Verify the source dataset's structure, available columns, event types, and suitability for customer funnel analysis.

**Processing approach:** Inspect a limited number of rows first to avoid loading the full dataset into memory.

In [1]:

from pathlib import Path
import pandas as pd

# Location of the extracted October CSV
csv_path = Path(
    r"C:\Users\kunal\Downloads\2019-Oct.csv\2019-Oct.csv"
)

# Confirm the file exists
print("File exists:", csv_path.is_file())

if not csv_path.is_file():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

# Read only five records, not the full dataset
df_preview = pd.read_csv(csv_path, nrows=5)

print("\nDataset columns:")
print(df_preview.columns.tolist())

print("\nPreview shape:")
print(df_preview.shape)

print("\nFirst five records:")
display(df_preview)


File exists: True

Dataset columns:
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']

Preview shape:
(5, 9)

First five records:


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


## Initial Data Quality and Event Inspection

This section examines the first 100,000 records to understand the dataset's structure, event types, missing values, and preliminary data quality.

The sample is used for exploratory inspection only. It is not assumed to represent the full October 2019 dataset.

In [2]:

# Read a limited exploratory sample
df_sample = pd.read_csv(csv_path, nrows=100_000)

print("Sample shape:")
print(df_sample.shape)

print("\nColumn data types:")
print(df_sample.dtypes)

print("\nEvent type counts:")
print(df_sample["event_type"].value_counts(dropna=False))

print("\nMissing values by column:")
print(df_sample.isna().sum())

print("\nMissing percentage by column:")
print((df_sample.isna().mean() * 100).round(2))

print("\nUnique users:")
print(df_sample["user_id"].nunique(dropna=True))

print("\nUnique sessions:")
print(df_sample["user_session"].nunique(dropna=True))


Sample shape:
(100000, 9)

Column data types:
event_time           str
event_type           str
product_id         int64
category_id        int64
category_code        str
brand                str
price            float64
user_id            int64
user_session         str
dtype: object

Event type counts:
event_type
view        97130
purchase     1655
cart         1215
Name: count, dtype: int64

Missing values by column:
event_time           0
event_type           0
product_id           0
category_id          0
category_code    32587
brand            14393
price                0
user_id              0
user_session         0
dtype: int64

Missing percentage by column:
event_time        0.00
event_type        0.00
product_id        0.00
category_id       0.00
category_code    32.59
brand            14.39
price             0.00
user_id           0.00
user_session      0.00
dtype: float64

Unique users:
20384

Unique sessions:
24382


## Full Dataset Profiling — Chunk Processing Test

The October 2019 dataset is too large to load entirely into memory.

We will process it in smaller chunks to support full-period validation while controlling memory usage.

In [3]:
    
# Test memory-efficient chunk processing
chunk_size = 100_000

reader = pd.read_csv(
    csv_path,
    chunksize=chunk_size
)

for chunk_number, chunk in enumerate(reader, start=1):
    print(f"Chunk number: {chunk_number}")
    print(f"Rows in chunk: {len(chunk):,}")
    print(f"Columns: {len(chunk.columns)}")

    # Stop after the first chunk for this test
    break

print("\nChunk processing test completed successfully.")


Chunk number: 1
Rows in chunk: 100,000
Columns: 9

Chunk processing test completed successfully.


## Full October Dataset Profiling

This section profiles the complete October 2019 e-commerce event dataset using chunk-based processing.

The objective is to calculate verified event frequencies, missing-value totals, and timestamp coverage without loading the entire CSV into memory.

The results will help validate the dataset before selecting a reproducible analytical sample.

In [4]:

import pandas as pd
from collections import Counter
from time import perf_counter

# Configuration
chunk_size = 100_000

# Profiling accumulators
total_rows = 0
chunk_count = 0
event_counts = Counter()
missing_counts = Counter()

earliest_event = None
latest_event = None

start_time = perf_counter()

# Process the entire October CSV
for chunk in pd.read_csv(csv_path, chunksize=chunk_size):

    chunk_count += 1
    total_rows += len(chunk)

    # Count event types
    event_counts.update(
        chunk["event_type"].fillna("MISSING").value_counts().to_dict()
    )

    # Count missing values in each column
    missing_counts.update(chunk.isna().sum().to_dict())

    # Find timestamp boundaries
    valid_times = chunk["event_time"].dropna()

    if not valid_times.empty:
        chunk_min = valid_times.min()
        chunk_max = valid_times.max()

        if earliest_event is None or chunk_min < earliest_event:
            earliest_event = chunk_min

        if latest_event is None or chunk_max > latest_event:
            latest_event = chunk_max

    if chunk_count % 50 == 0:
        print(f"Processed {total_rows:,} rows...")

elapsed = perf_counter() - start_time

print("\n========== FULL DATASET PROFILE ==========")
print(f"Total rows: {total_rows:,}")
print(f"Chunks processed: {chunk_count:,}")
print(f"Processing time: {elapsed:.1f} seconds")

print("\nEvent counts:")
for event, count in event_counts.most_common():
    print(f"{event}: {count:,}")

print("\nMissing values:")
for column, count in missing_counts.items():
    percentage = count / total_rows * 100
    print(f"{column}: {count:,} ({percentage:.2f}%)")

print("\nTimestamp coverage:")
print("Earliest:", earliest_event)
print("Latest:", latest_event)


Processed 5,000,000 rows...
Processed 10,000,000 rows...
Processed 15,000,000 rows...
Processed 20,000,000 rows...
Processed 25,000,000 rows...
Processed 30,000,000 rows...
Processed 35,000,000 rows...
Processed 40,000,000 rows...

========== FULL DATASET PROFILE ==========
Total rows: 42,448,764
Chunks processed: 425
Processing time: 57.8 seconds

Event counts:
view: 40,779,399
cart: 926,516
purchase: 742,849

Missing values:
event_time: 0 (0.00%)
event_type: 0 (0.00%)
product_id: 0 (0.00%)
category_id: 0 (0.00%)
category_code: 13,515,609 (31.84%)
brand: 6,117,080 (14.41%)
price: 0 (0.00%)
user_id: 0 (0.00%)
user_session: 2 (0.00%)

Timestamp coverage:
Earliest: 2019-10-01 00:00:00 UTC
Latest: 2019-10-31 23:59:59 UTC


In [5]:

# Display complete missing-value summary
missing_summary = pd.DataFrame({
    "column": list(missing_counts.keys()),
    "missing_count": list(missing_counts.values())
})

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / total_rows * 100
).round(2)

display(missing_summary)


,column,missing_count,missing_percentage
0,event_time,0,0.00
1,event_type,0,0.00
2,product_id,0,0.00
3,category_id,0,0.00
4,category_code,13515609,31.84
5,brand,6117080,14.41
6,price,0,0.00
7,user_id,0,0.00
8,user_session,2,0.00


## Export Full Dataset Profiling Results

Save the verified full-file profiling results to JSON for reproducibility, documentation, and later validation.

In [6]:

import json
from pathlib import Path

# Find project root from the notebook's working directory
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

output_path = project_root / "data" / "october_profile.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

profile = {
    "dataset": "2019-Oct.csv",
    "total_rows": total_rows,
    "chunks_processed": chunk_count,
    "event_counts": dict(event_counts),
    "missing_counts": dict(missing_counts),
    "earliest_event": earliest_event,
    "latest_event": latest_event,
    "processing_seconds": round(elapsed, 2)
}

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(profile, file, indent=4)

print("Profile saved to:", output_path)
print("File exists:", output_path.is_file())


Profile saved to: c:\Users\kunal\OneDrive\Desktop\product-funnel-analysis\data\october_profile.json
File exists: True


## Deterministic Customer Sampling — Logic Test

Test a stable customer-level sampling rule before applying it to the complete October dataset.

All events belonging to a selected customer should receive the same selection decision.

In [7]:

import hashlib

def select_customer(user_id, sample_rate=0.02):
    # Stable hash for reproducible customer selection
    hashed = hashlib.sha256(
        str(int(user_id)).encode("utf-8")
    ).digest()

    hash_value = int.from_bytes(hashed[:8], "big")

    # Map the hash into a value between 0 and 1
    normalized = hash_value / (2 ** 64)

    return normalized < sample_rate


# Test on our existing 100,000-row inspection sample
test_sample = df_sample.copy()

test_sample["selected"] = (
    test_sample["user_id"].map(select_customer)
)

print("Rows inspected:", len(test_sample))
print("Selected rows:", test_sample["selected"].sum())

print(
    "Selected unique customers:",
    test_sample.loc[
        test_sample["selected"], "user_id"
    ].nunique()
)

# Verify one selection decision per customer
selection_consistency = (
    test_sample.groupby("user_id")["selected"]
    .nunique()
    .max()
)

print(
    "Maximum selection decisions per customer:",
    selection_consistency
)


Rows inspected: 100000
Selected rows: 2147
Selected unique customers: 441
Maximum selection decisions per customer: 1


## Full October Customer-Level Sampling

Apply the deterministic 2% customer-selection rule across the complete October dataset.

Retain every event belonging to selected customers, preserving their observed October event histories.

The resulting dataset will be used for subsequent validation and analytical preparation.

In [8]:

import pandas as pd
from pathlib import Path
from time import perf_counter

# Output location
sample_path = project_root / "data" / "raw" / "october_customer_sample.csv"
sample_path.parent.mkdir(parents=True, exist_ok=True)

chunk_size = 100_000

total_scanned = 0
total_selected = 0
chunks_processed = 0

start_time = perf_counter()

# Create a fresh output file on each run
if sample_path.exists():
    sample_path.unlink()

for chunk in pd.read_csv(csv_path, chunksize=chunk_size):

    chunks_processed += 1
    total_scanned += len(chunk)

    # Select customers using the stable hash rule
    selected_mask = chunk["user_id"].map(select_customer)

    selected_chunk = chunk.loc[selected_mask]

    if not selected_chunk.empty:
        selected_chunk.to_csv(
            sample_path,
            mode="a",
            index=False,
            header=not sample_path.exists()
        )

        total_selected += len(selected_chunk)

    if chunks_processed % 50 == 0:
        print(f"Scanned {total_scanned:,} events...")

elapsed_sampling = perf_counter() - start_time

print("\n========== SAMPLING SUMMARY ==========")
print(f"Total scanned: {total_scanned:,}")
print(f"Total selected: {total_selected:,}")
print(f"Selection share: {total_selected / total_scanned * 100:.2f}%")
print(f"Chunks processed: {chunks_processed}")
print(f"Processing time: {elapsed_sampling:.1f} seconds")
print(f"Sample saved to: {sample_path}")
print(f"Sample exists: {sample_path.is_file()}")


Scanned 5,000,000 events...
Scanned 10,000,000 events...
Scanned 15,000,000 events...
Scanned 20,000,000 events...
Scanned 25,000,000 events...
Scanned 30,000,000 events...
Scanned 35,000,000 events...
Scanned 40,000,000 events...

========== SAMPLING SUMMARY ==========
Total scanned: 42,448,764
Total selected: 849,220
Selection share: 2.00%
Chunks processed: 425
Processing time: 117.3 seconds
Sample saved to: c:\Users\kunal\OneDrive\Desktop\product-funnel-analysis\data\raw\october_customer_sample.csv
Sample exists: True


## Analytical Sample Validation

Validate the deterministic customer-level sample against the full October dataset profile.

Check sample size, unique identifiers, timestamp coverage, event distribution, and differences from the full dataset.

Event frequencies are not customer conversion rates.

In [9]:

import pandas as pd
from collections import Counter
from time import perf_counter

# Configuration
chunk_size = 100_000

# Profiling accumulators
total_rows = 0
chunk_count = 0
event_counts = Counter()
missing_counts = Counter()

earliest_event = None
latest_event = None

start_time = perf_counter()

# Process the entire October CSV
for chunk in pd.read_csv(csv_path, chunksize=chunk_size):

    chunk_count += 1
    total_rows += len(chunk)

    # Count event types
    event_counts.update(
        chunk["event_type"].fillna("MISSING").value_counts().to_dict()
    )

    # Count missing values in each column
    missing_counts.update(chunk.isna().sum().to_dict())

    # Find timestamp boundaries
    valid_times = chunk["event_time"].dropna()

    if not valid_times.empty:
        chunk_min = valid_times.min()
        chunk_max = valid_times.max()

        if earliest_event is None or chunk_min < earliest_event:
            earliest_event = chunk_min

        if latest_event is None or chunk_max > latest_event:
            latest_event = chunk_max

    if chunk_count % 50 == 0:
        print(f"Processed {total_rows:,} rows...")

elapsed = perf_counter() - start_time

print("\n========== FULL DATASET PROFILE ==========")
print(f"Total rows: {total_rows:,}")
print(f"Chunks processed: {chunk_count:,}")
print(f"Processing time: {elapsed:.1f} seconds")

print("\nEvent counts:")
for event, count in event_counts.most_common():
    print(f"{event}: {count:,}")

print("\nMissing values:")
for column, count in missing_counts.items():
    percentage = count / total_rows * 100
    print(f"{column}: {count:,} ({percentage:.2f}%)")

print("\nTimestamp coverage:")
print("Earliest:", earliest_event)
print("Latest:", latest_event)


Processed 5,000,000 rows...
Processed 10,000,000 rows...
Processed 15,000,000 rows...
Processed 20,000,000 rows...
Processed 25,000,000 rows...
Processed 30,000,000 rows...
Processed 35,000,000 rows...
Processed 40,000,000 rows...

========== FULL DATASET PROFILE ==========
Total rows: 42,448,764
Chunks processed: 425
Processing time: 55.9 seconds

Event counts:
view: 40,779,399
cart: 926,516
purchase: 742,849

Missing values:
event_time: 0 (0.00%)
event_type: 0 (0.00%)
product_id: 0 (0.00%)
category_id: 0 (0.00%)
category_code: 13,515,609 (31.84%)
brand: 6,117,080 (14.41%)
price: 0 (0.00%)
user_id: 0 (0.00%)
user_session: 2 (0.00%)

Timestamp coverage:
Earliest: 2019-10-01 00:00:00 UTC
Latest: 2019-10-31 23:59:59 UTC


In [10]:

# Step 12C — Analytical Sample Validation

df_analytical = pd.read_csv(sample_path)

print("========== SAMPLE VALIDATION ==========")

print(f"Total sample rows: {len(df_analytical):,}")
print(f"Unique customers: {df_analytical['user_id'].nunique():,}")
print(f"Unique sessions: {df_analytical['user_session'].nunique():,}")

print("\nTimestamp coverage:")
print("Earliest:", df_analytical["event_time"].min())
print("Latest:", df_analytical["event_time"].max())

print("\nSample event counts:")
print(df_analytical["event_type"].value_counts())

# Compare sample distribution with full dataset
full_distribution = (
    pd.Series(dict(event_counts)) / total_rows * 100
)

sample_distribution = (
    df_analytical["event_type"].value_counts(normalize=True) * 100
)

comparison = pd.DataFrame({
    "full_dataset_pct": full_distribution,
    "sample_pct": sample_distribution
}).fillna(0)

comparison["difference_pp"] = (
    comparison["sample_pct"] - comparison["full_dataset_pct"]
)

print("\nEvent distribution comparison:")
display(comparison.round(3))

print("\nSample missing values:")
display(df_analytical.isna().sum())


========== SAMPLE VALIDATION ==========
Total sample rows: 849,220
Unique customers: 60,939
Unique sessions: 185,065

Timestamp coverage:
Earliest: 2019-10-01 00:00:56 UTC
Latest: 2019-10-31 23:57:44 UTC

Sample event counts:
event_type
view        814847
cart         18938
purchase     15435
Name: count, dtype: int64

Event distribution comparison:


,full_dataset_pct,sample_pct,difference_pp
cart,2.183,2.230,0.047
purchase,1.750,1.818,0.068
view,96.067,95.952,-0.115



Sample missing values:


event_time            0
event_type            0
product_id            0
category_id           0
category_code    268555
brand            121933
price                 0
user_id               0
user_session          0
dtype: int64

## Daily Coverage Validation

Verify that the customer-level analytical sample contains events across all 31 days of October 2019.

Inspect daily event counts to identify missing days and unusually low or high activity.

In [11]:

import pandas as pd
from collections import Counter
from time import perf_counter

# Configuration
chunk_size = 100_000

# Profiling accumulators
total_rows = 0
chunk_count = 0
event_counts = Counter()
missing_counts = Counter()

earliest_event = None
latest_event = None

start_time = perf_counter()

# Process the entire October CSV
for chunk in pd.read_csv(csv_path, chunksize=chunk_size):

    chunk_count += 1
    total_rows += len(chunk)

    # Count event types
    event_counts.update(
        chunk["event_type"].fillna("MISSING").value_counts().to_dict()
    )

    # Count missing values in each column
    missing_counts.update(chunk.isna().sum().to_dict())

    # Find timestamp boundaries
    valid_times = chunk["event_time"].dropna()

    if not valid_times.empty:
        chunk_min = valid_times.min()
        chunk_max = valid_times.max()

        if earliest_event is None or chunk_min < earliest_event:
            earliest_event = chunk_min

        if latest_event is None or chunk_max > latest_event:
            latest_event = chunk_max

    if chunk_count % 50 == 0:
        print(f"Processed {total_rows:,} rows...")

elapsed = perf_counter() - start_time

print("\n========== FULL DATASET PROFILE ==========")
print(f"Total rows: {total_rows:,}")
print(f"Chunks processed: {chunk_count:,}")
print(f"Processing time: {elapsed:.1f} seconds")

print("\nEvent counts:")
for event, count in event_counts.most_common():
    print(f"{event}: {count:,}")

print("\nMissing values:")
for column, count in missing_counts.items():
    percentage = count / total_rows * 100
    print(f"{column}: {count:,} ({percentage:.2f}%)")

print("\nTimestamp coverage:")
print("Earliest:", earliest_event)
print("Latest:", latest_event)


Processed 5,000,000 rows...
Processed 10,000,000 rows...
Processed 15,000,000 rows...
Processed 20,000,000 rows...
Processed 25,000,000 rows...
Processed 30,000,000 rows...
Processed 35,000,000 rows...
Processed 40,000,000 rows...

========== FULL DATASET PROFILE ==========
Total rows: 42,448,764
Chunks processed: 425
Processing time: 54.9 seconds

Event counts:
view: 40,779,399
cart: 926,516
purchase: 742,849

Missing values:
event_time: 0 (0.00%)
event_type: 0 (0.00%)
product_id: 0 (0.00%)
category_id: 0 (0.00%)
category_code: 13,515,609 (31.84%)
brand: 6,117,080 (14.41%)
price: 0 (0.00%)
user_id: 0 (0.00%)
user_session: 2 (0.00%)

Timestamp coverage:
Earliest: 2019-10-01 00:00:00 UTC
Latest: 2019-10-31 23:59:59 UTC


In [13]:

import pandas as pd

# Load the existing analytical sample if needed
if "df_analytical" not in globals():
    df_analytical = pd.read_csv(
        "data/raw/october_customer_sample.csv"
    )

# Convert timestamps to datetime
df_analytical["event_time"] = pd.to_datetime(
    df_analytical["event_time"],
    utc=True
)

# Count events for each day
daily_counts = (
    df_analytical
    .groupby(df_analytical["event_time"].dt.date)
    .size()
)

# Create the expected October dates
expected_dates = pd.date_range(
    "2019-10-01",
    "2019-10-31",
    freq="D"
).date

# Identify missing dates
missing_dates = sorted(
    set(expected_dates) - set(daily_counts.index)
)

print("========== DAILY COVERAGE CHECK ==========")
print("Days represented:", len(daily_counts))
print("First day:", daily_counts.index.min())
print("Last day:", daily_counts.index.max())
print("Missing dates:", missing_dates)
print("All 31 days covered:", len(missing_dates) == 0)


========== DAILY COVERAGE CHECK ==========
Days represented: 31
First day: 2019-10-01
Last day: 2019-10-31
Missing dates: []
All 31 days covered: True


## Customer Sampling Integrity Validation

Verify that all customers in the analytical sample satisfy the deterministic SHA-256 selection rule.

This validation checks customer eligibility and identifier completeness. Full event-retention validation will be performed separately.

In [14]:

# Step 12E — Customer Sampling Integrity

# Extract unique customer IDs
sample_customers = df_analytical["user_id"].dropna().unique()

# Apply our original selection function
customer_checks = pd.Series(sample_customers).map(
    select_customer
)

print("========== CUSTOMER SAMPLING INTEGRITY ==========")

print(f"Unique customers checked: {len(sample_customers):,}")

print(
    "Customers passing selection rule:",
    int(customer_checks.sum())
)

print(
    "Customers failing selection rule:",
    int((~customer_checks).sum())
)

print(
    "Missing customer IDs:",
    int(df_analytical["user_id"].isna().sum())
)

print(
    "Sampling rule validation passed:",
    bool(customer_checks.all())
)


========== CUSTOMER SAMPLING INTEGRITY ==========
Unique customers checked: 60,939
Customers passing selection rule: 60939
Customers failing selection rule: 0
Missing customer IDs: 0
Sampling rule validation passed: True


## Full Event-Retention Integrity Validation

Independently verify that the analytical sample retains every event from the original October dataset for each selected customer.

Compare source event counts against sampled event counts at the customer level.

In [15]:

from collections import Counter
from time import perf_counter

# Selected customer IDs from the saved sample
selected_customer_ids = set(
    df_analytical["user_id"].dropna().unique()
)

# Count events per customer in our analytical sample
sample_customer_counts = Counter(
    df_analytical["user_id"].value_counts().to_dict()
)

# Independently count matching events in the original CSV
source_customer_counts = Counter()

source_matching_events = 0
source_rows_scanned = 0

start = perf_counter()

for chunk in pd.read_csv(
    csv_path,
    usecols=["user_id"],
    chunksize=100_000
):
    source_rows_scanned += len(chunk)

    matching_ids = chunk.loc[
        chunk["user_id"].isin(selected_customer_ids),
        "user_id"
    ]

    source_matching_events += len(matching_ids)

    source_customer_counts.update(
        matching_ids.value_counts().to_dict()
    )

elapsed_validation = perf_counter() - start

# Compare counts for every selected customer
mismatched_customers = [
    user_id
    for user_id in selected_customer_ids
    if source_customer_counts[user_id]
    != sample_customer_counts[user_id]
]

validation_passed = (
    source_rows_scanned == total_rows
    and source_matching_events == len(df_analytical)
    and len(mismatched_customers) == 0
)

print("========== EVENT RETENTION VALIDATION ==========")
print(f"Source rows scanned: {source_rows_scanned:,}")
print(f"Selected customers checked: {len(selected_customer_ids):,}")
print(f"Matching source events: {source_matching_events:,}")
print(f"Saved sample events: {len(df_analytical):,}")
print(f"Customers with count mismatches: {len(mismatched_customers):,}")
print(f"Validation time: {elapsed_validation:.1f} seconds")
print(f"Full event-retention validation passed: {validation_passed}")


========== EVENT RETENTION VALIDATION ==========
Source rows scanned: 42,448,764
Selected customers checked: 60,939
Matching source events: 849,220
Saved sample events: 849,220
Customers with count mismatches: 0
Validation time: 18.9 seconds
Full event-retention validation passed: True


## Step 14B — Funnel Event Readiness

Examine which funnel event types appear in the analytical sample
and how many unique customers generate each event type.

These are descriptive event counts, not sequential funnel
conversion rates.

In [16]:

# Step 14B — Funnel Event Readiness

event_summary = (
    df_analytical
    .groupby("event_type")
    .agg(
        total_events=("user_id", "size"),
        unique_customers=("user_id", "nunique"),
        unique_sessions=("user_session", "nunique")
    )
    .sort_values("total_events", ascending=False)
)

event_summary["event_share_pct"] = (
    event_summary["total_events"]
    / len(df_analytical) * 100
).round(2)

print("========== FUNNEL EVENT READINESS ==========")
print(event_summary.to_string())

print("\nTotal unique customers:",
      df_analytical["user_id"].nunique())

print("Total unique sessions:",
      df_analytical["user_session"].nunique())


========== FUNNEL EVENT READINESS ==========
            total_events  unique_customers  unique_sessions  event_share_pct
event_type                                                                  
view              814847             60936           185022            95.95
cart               18938              6781            11743             2.23
purchase           15435              7099            12993             1.82

Total unique customers: 60939
Total unique sessions: 185065


## Step 14C — Customer Event Overlap

Identify which event types each customer performed during October.

This analysis checks event presence, not chronological order.

In [17]:

# Step 14C — Customer Event Overlap

customer_events = pd.crosstab(
    df_analytical["user_id"],
    df_analytical["event_type"]
).gt(0)

for event in ["view", "cart", "purchase"]:
    if event not in customer_events.columns:
        customer_events[event] = False

has_view = customer_events["view"]
has_cart = customer_events["cart"]
has_purchase = customer_events["purchase"]

print("========== CUSTOMER EVENT OVERLAP ==========")
print("Customers with views:", int(has_view.sum()))
print("Customers with carts:", int(has_cart.sum()))
print("Customers with purchases:", int(has_purchase.sum()))

print("\nCustomers with both view and cart:",
      int((has_view & has_cart).sum()))

print("Customers with both cart and purchase:",
      int((has_cart & has_purchase).sum()))

print("Customers with all three event types:",
      int((has_view & has_cart & has_purchase).sum()))

print("Customers with purchase but NO cart:",
      int((has_purchase & ~has_cart).sum()))

print("Customers with purchase but NO view:",
      int((has_purchase & ~has_view).sum()))

print("\nTotal customers checked:",
      len(customer_events))


========== CUSTOMER EVENT OVERLAP ==========
Customers with views: 60936
Customers with carts: 6781
Customers with purchases: 7099

Customers with both view and cart: 6778
Customers with both cart and purchase: 4124
Customers with all three event types: 4123
Customers with purchase but NO cart: 2975
Customers with purchase but NO view: 1

Total customers checked: 60939


## Step 14D — Customer-Level Event Sequence

Check whether each customer has an observed chronological
view → cart → purchase sequence.

This is a preliminary customer-level diagnostic.
Product-level and session-level consistency will be
evaluated before finalizing the funnel definition.

In [18]:

# Step 14D — Chronological Customer Funnel

events = df_analytical[
    ["user_id", "event_time", "event_type"]
].copy()

events["event_time"] = pd.to_datetime(
    events["event_time"], utc=True
)

# Earliest observed view for each customer
first_views = (
    events.loc[events["event_type"] == "view"]
    .groupby("user_id")["event_time"]
    .min()
    .rename("first_view")
)

# Find cart events occurring after a view
carts = events.loc[
    events["event_type"] == "cart",
    ["user_id", "event_time"]
].join(first_views, on="user_id")

valid_carts = carts.loc[
    carts["event_time"] > carts["first_view"]
]

first_valid_carts = (
    valid_carts.groupby("user_id")["event_time"]
    .min()
    .rename("first_valid_cart")
)

# Find purchases occurring after a valid cart
purchases = events.loc[
    events["event_type"] == "purchase",
    ["user_id", "event_time"]
].join(first_valid_carts, on="user_id")

valid_purchases = purchases.loc[
    purchases["event_time"] > purchases["first_valid_cart"]
]

print("========== CHRONOLOGICAL FUNNEL ==========")
print("Customers with view:", len(first_views))
print("Customers with view -> cart:",
      len(first_valid_carts))
print("Customers with view -> cart -> purchase:",
      valid_purchases["user_id"].nunique())

print("\nCustomers with all three event types:",
      int((
          customer_events["view"]
          & customer_events["cart"]
          & customer_events["purchase"]
      ).sum()))


========== CHRONOLOGICAL FUNNEL ==========
Customers with view: 60936
Customers with view -> cart: 6771
Customers with view -> cart -> purchase: 4004

Customers with all three event types: 4123


## Step 14E — Same-Product Funnel Validation

Validate chronological view → cart → purchase sequences
using both customer_id and product_id.

This prevents unrelated products from being combined
into one product conversion journey.

Events may still occur across different sessions.

In [19]:

# Step 14E — Same-Product Chronological Funnel

product_events = df_analytical[
    ["user_id", "product_id", "event_time", "event_type"]
].copy()

product_events["event_time"] = pd.to_datetime(
    product_events["event_time"],
    utc=True
)

keys = ["user_id", "product_id"]

# Earliest observed view per customer-product pair
product_views = (
    product_events.loc[
        product_events["event_type"] == "view"
    ]
    .groupby(keys)["event_time"]
    .min()
    .rename("first_view")
)

# Cart must occur after a view of the same product
product_carts = (
    product_events.loc[
        product_events["event_type"] == "cart",
        keys + ["event_time"]
    ]
    .join(product_views, on=keys)
)

valid_product_carts = product_carts.loc[
    product_carts["event_time"]
    > product_carts["first_view"]
]

first_product_carts = (
    valid_product_carts
    .groupby(keys)["event_time"]
    .min()
    .rename("first_valid_cart")
)

# Purchase must occur after a valid cart
# for the same customer-product pair
product_purchases = (
    product_events.loc[
        product_events["event_type"] == "purchase",
        keys + ["event_time"]
    ]
    .join(first_product_carts, on=keys)
)

valid_product_purchases = product_purchases.loc[
    product_purchases["event_time"]
    > product_purchases["first_valid_cart"]
]

# Distinct customers reaching each stage
product_view_customers = (
    product_views.index.get_level_values("user_id").nunique()
)

product_cart_customers = (
    first_product_carts.index.get_level_values("user_id").nunique()
)

product_purchase_customers = (
    valid_product_purchases["user_id"].nunique()
)

print("========== SAME-PRODUCT FUNNEL ==========")
print("Customers with product view:",
      product_view_customers)

print("Customers with same-product view -> cart:",
      product_cart_customers)

print("Customers with same-product view -> cart -> purchase:",
      product_purchase_customers)

print("\nCustomer-level chronological completions:",
      4004)

print("Same-product chronological completions:",
      product_purchase_customers)


========== SAME-PRODUCT FUNNEL ==========
Customers with product view: 60936
Customers with same-product view -> cart: 6755
Customers with same-product view -> cart -> purchase: 3772

Customer-level chronological completions: 4004
Same-product chronological completions: 3772


## Step 14F — Same-Session, Same-Product Funnel

Check chronological view → cart → purchase sequences
within the same customer, session, and product.

Compare this stricter funnel with the customer-product
funnel from Step 14E.

In [20]:

# Step 14F — Same-Session, Same-Product Funnel

session_events = df_analytical[
    [
        "user_id",
        "user_session",
        "product_id",
        "event_time",
        "event_type"
    ]
].dropna(subset=["user_session"]).copy()

session_events["event_time"] = pd.to_datetime(
    session_events["event_time"],
    utc=True
)

keys = ["user_id", "user_session", "product_id"]

# Earliest view per customer-session-product
session_views = (
    session_events.loc[
        session_events["event_type"] == "view"
    ]
    .groupby(keys)["event_time"]
    .min()
    .rename("first_view")
)

# Cart after view within the same group
session_carts = (
    session_events.loc[
        session_events["event_type"] == "cart",
        keys + ["event_time"]
    ]
    .join(session_views, on=keys)
)

valid_session_carts = session_carts.loc[
    session_carts["event_time"]
    > session_carts["first_view"]
]

first_session_carts = (
    valid_session_carts
    .groupby(keys)["event_time"]
    .min()
    .rename("first_valid_cart")
)

# Purchase after qualifying cart
session_purchases = (
    session_events.loc[
        session_events["event_type"] == "purchase",
        keys + ["event_time"]
    ]
    .join(first_session_carts, on=keys)
)

valid_session_purchases = session_purchases.loc[
    session_purchases["event_time"]
    > session_purchases["first_valid_cart"]
]

# Count unique customers
session_view_customers = (
    session_views.index.get_level_values("user_id").nunique()
)

session_cart_customers = (
    first_session_carts.index.get_level_values("user_id").nunique()
)

session_purchase_customers = (
    valid_session_purchases["user_id"].nunique()
)

print("========== SAME-SESSION SAME-PRODUCT FUNNEL ==========")

print("Customers with view:",
      session_view_customers)

print("Customers with view -> cart:",
      session_cart_customers)

print("Customers with view -> cart -> purchase:",
      session_purchase_customers)

print("\nSame-product completions across sessions:", 3772)

print("Same-session same-product completions:",
      session_purchase_customers)

print("\nQualifying customer-session-product journeys:",
      valid_session_purchases[keys].drop_duplicates().shape[0])


========== SAME-SESSION SAME-PRODUCT FUNNEL ==========
Customers with view: 60936
Customers with view -> cart: 6742
Customers with view -> cart -> purchase: 3640

Same-product completions across sessions: 3772
Same-session same-product completions: 3640

Qualifying customer-session-product journeys: 6300


## Step 15A — Journey-Level Funnel Counts

Count distinct customer-session-product combinations reaching
each chronological funnel stage.

These counts will provide consistent denominators for
journey-level conversion and drop-off KPIs.

In [21]:

# Step 15A — Journey-Level Funnel Counts

# The objects below were created in Step 14F.
# Each index represents one customer-session-product journey.

view_journeys = len(session_views)

cart_journeys = len(first_session_carts)

purchase_journeys = (
    valid_session_purchases[keys]
    .drop_duplicates()
    .shape[0]
)

print("========== JOURNEY-LEVEL FUNNEL ==========")
print(f"View journeys: {view_journeys:,}")
print(f"View -> Cart journeys: {cart_journeys:,}")
print(f"View -> Cart -> Purchase journeys: {purchase_journeys:,}")

print("\n========== JOURNEY CONVERSION RATES ==========")

if view_journeys > 0:
    print(
        f"View -> Cart: {cart_journeys / view_journeys * 100:.2f}%"
    )

if cart_journeys > 0:
    print(
        f"Cart -> Purchase: {purchase_journeys / cart_journeys * 100:.2f}%"
    )

if view_journeys > 0:
    print(
        f"View -> Purchase: {purchase_journeys / view_journeys * 100:.2f}%"
    )

print("\n========== JOURNEY DROP-OFF ==========")
print(f"View -> Cart drop-off: {view_journeys - cart_journeys:,}")
print(f"Cart -> Purchase drop-off: {cart_journeys - purchase_journeys:,}")


========== JOURNEY-LEVEL FUNNEL ==========
View journeys: 552,900
View -> Cart journeys: 12,725
View -> Cart -> Purchase journeys: 6,300

========== JOURNEY CONVERSION RATES ==========
View -> Cart: 2.30%
Cart -> Purchase: 49.51%
View -> Purchase: 1.14%

========== JOURNEY DROP-OFF ==========
View -> Cart drop-off: 540,175
Cart -> Purchase drop-off: 6,425
